# Second Model - Random Forest Regressor

**Why this model?**

Our research question is not which model predicts best, but whether the gap between random, blocked and chronological validation depends on the kind of model. Ridge and gradient boosting sit at two ends of that question: one is a rigid linear model, the other a flexible tree ensemble that corrects its own errors step by step. A random forest fills the gap between them with a different mechanism.

A forest grows many deep trees independently, each on a bootstrap sample of rows and with a random subset of features at every split, and averages their predictions. Averaging cuts the variance of individual trees, whereas boosting reduces bias. Like boosting, it finds the interaction between hour and day type on its own and needs no manual feature engineering. Unlike boosting, it has few sensitive hyperparameters and does not overfit as more trees are added, which makes it a more robust second tree-based model.

The main reason to include it is memorisation. With small leaves a forest can almost reproduce the training data, so under random k-fold it can lean heavily on neighbouring hours that sit in both train and test. If the gap between schemes is a general property of flexible models, the forest should show it as well. If it is specific to boosting, the forest should behave differently. We also expect the chosen min_samples_leaf to reflect the scheme: random validation should favour smaller leaves than the chronological one.

The forest shares one limitation with boosting: trees cannot extrapolate beyond the range of the training targets, so we expect demand in 2012, which is higher than in 2011, to be underestimated. We handle the missing values of hum with median imputation inside the pipeline, so nothing from the validation part leaks into the imputer.

In [1]:
import ast
import time

import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from src.data import load_data
from src.validation import RESULTS_DIR, SCHEMES, SEED, evaluate, get_cv

In [2]:
X, y = load_data(drop__yr=True)

assert X.index.equals(y.index)
assert X.index.is_monotonic_increasing
assert X.index.is_unique
assert "yr" not in X.columns
assert X["hum"].isna().sum() == 22

print("X shape:", X.shape, "| y shape:", y.shape)
print("Features:", X.columns.tolist())
print("Validation schemes:", SCHEMES)

X shape: (17379, 9) | y shape: (17379,)
Features: ['mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']
Validation schemes: ('random', 'blocked', 'chrono')


## Preprocessing

Unlike linear models, tree-based ensembles are invariant to monotonic feature scaling and do not require one-hot encoding of categorical indicators. The model directly ingests the processed tabular features (`mnth`, `hr`, `holiday`, `weekday`, `workingday`, `weathersit`, `temp`, `hum`, `windspeed`). 

The 22 missing values in `hum` are imputed using `SimpleImputer(strategy="median")` encapsulated within a scikit-learn `Pipeline`. To guarantee strict methodological consistency with our Ridge setup and prevent test leakage, the imputation median is calculated exclusively on the training folds of each split. While modern implementations occasionally offer native missing-value routing, utilizing an explicit pipeline ensures deterministic and leak-free transformations across all outer and inner validation folds.

In [3]:
N_ESTIMATORS = 200

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=SEED, n_jobs=1)),
])

## Random Forest and Hyperparameters

Hyperparameter selection focuses on governing tree complexity and ensemble decorrelation:

- `n_estimators`: Fixed at 200 trees. Ensembles governed by bagging do not overfit as tree count increases; adding trees monotonically reduces estimation variance at the expense of marginal compute time.
- `min_samples_leaf`: Serves as the primary complexity and smoothing control (`[1, 5, 20]`). A value of 1 permits individual hour memorization, while higher thresholds prevent trees from isolating noisy temporal spikes.
- `max_features`: Controls feature subspace sampling (`[0.5, 1.0]`). Evaluating a random subset (50%) at each split decorrelates individual trees, boosting ensemble robustness.
- `max_depth`: Left unconstrained (`None`), as regularization is governed explicitly via `min_samples_leaf`.
- `oob_score`: Deliberately set to `False`. Out-of-bag scoring relies on random row-level bootstrapping, which violates the strict non-overlapping temporal partitioning of chronological splits.

In [4]:
PARAM_GRID = {
    "min_samples_leaf": [1, 5, 20],
    "max_features": [0.5, 1.0],
}
INNER_SPLITS = 3


def make_rf_search(scheme, log_target=False):
    """RF pipeline tuned with an inner CV of the same type as the outer scheme."""
    if log_target:
        estimator = TransformedTargetRegressor(
            regressor=rf_pipeline, func=np.log1p, inverse_func=np.expm1
        )
        param_grid = {f"regressor__model__{k}": v for k, v in PARAM_GRID.items()}
    else:
        estimator = rf_pipeline
        param_grid = {f"model__{k}": v for k, v in PARAM_GRID.items()}

    return GridSearchCV(
        estimator,
        param_grid,
        cv=get_cv(scheme, n_splits=INNER_SPLITS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )

## Computational Cost

A single hyperparameter search over 6 candidate configurations ($3 \times 2$) with 3 inner cross-validation folds executes in approximately 26.6 seconds on the complete dataset. Extrapolating across 5 outer folds and 3 validation schemes yields approximately 7 to 8 minutes of total compute time per model variant, confirming that the parameter grid and 200-estimator depth are computationally practical.

In [5]:
start = time.perf_counter()
benchmark = make_rf_search("random").fit(X, y)
print(f"One search on all data: {time.perf_counter() - start:.1f} s")
print("Best parameters:", benchmark.best_params_)

One search on all data: 26.6 s
Best parameters: {'model__max_features': 0.5, 'model__min_samples_leaf': 1}


## Validation Strategy and Hyperparameter Tuning

We implement a strictly nested cross-validation protocol using the three core evaluation schemes defined in `src/validation.py`:

- **random**: Standard 5-fold cross-validation with random row shuffling. Adjacent temporal points appear across both training and validation splits.
- **blocked**: Contiguous non-overlapping temporal blocks evaluated sequentially. No shuffling is performed, though training sets include both preceding and subsequent observations.
- **chrono**: Forward-chaining expanding window over 2012 test blocks. The model is trained strictly on historical data prior to the evaluation window, simulating production forecasting.

Hyperparameters are optimized inside each outer training fold using `GridSearchCV` over 3 inner splits matched to the respective outer scheme type. Once the optimal parameter configuration is identified via negative root mean squared error, the pipeline is refitted on the entire outer training split and evaluated against the unseen outer test fold.

In [6]:
variants = {"rf": False}  

results = {}
for name, log_target in variants.items():
    for scheme in SCHEMES:
        start = time.perf_counter()
        model = make_rf_search(scheme, log_target)
        folds, preds = evaluate(model, X, y, scheme, name)
        results[(name, scheme)] = (folds, preds)
        print(f"{name:8s} {scheme:8s} {(time.perf_counter() - start) / 60:5.2f} min   "
              f"mean RMSE {folds['rmse'].mean():.1f}   mean MAE {folds['mae'].mean():.1f}")

rf       random    1.27 min   mean RMSE 65.8   mean MAE 42.9
rf       blocked   1.28 min   mean RMSE 114.1   mean MAE 80.6
rf       chrono    1.44 min   mean RMSE 117.1   mean MAE 83.1


## Experimental Results

The table below summarizes the performance metrics averaged across outer folds for both baseline estimators and the Random Forest regressor (`rf`). The metrics include Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), Coefficient of Determination ($R^2$), and Root Mean Squared Logarithmic Error (RMSLE). 

Because the chronological scheme evaluates strictly on 2012 test blocks whereas random and blocked cross-validations span the entirety of 2011–2012, fold-averaged metrics must only be used to rank competing models within a given scheme, rather than to measure the cross-validation gap directly.

In [7]:
def summarise(name, scheme):
    folds = pd.read_csv(RESULTS_DIR / f"{name}_{scheme}_folds.csv")
    return {
        "model": name,
        "scheme": scheme,
        "rmse_mean": folds["rmse"].mean(),
        "rmse_std": folds["rmse"].std(),
        "mae_mean": folds["mae"].mean(),
        "r2_mean": folds["r2"].mean(),
        "rmsle_mean": folds["rmsle"].mean(),
    }


names = ["baseline_const", "baseline_hour_day", *variants]
pd.DataFrame([summarise(n, s) for n in names for s in SCHEMES]).round(3)

,model,scheme,rmse_mean,rmse_std,mae_mean,r2_mean,rmsle_mean
0,baseline_const,random,181.372,3.540,142.415,-0.001,1.570
1,baseline_const,blocked,182.116,47.228,147.169,-0.280,1.575
2,baseline_const,chrono,215.611,52.739,165.455,-0.159,1.443
3,baseline_hour_day,random,109.942,1.384,72.959,0.632,0.648
4,baseline_hour_day,blocked,119.111,38.852,83.458,0.304,0.658
5,baseline_hour_day,chrono,133.292,44.825,95.482,0.562,0.580
6,rf,random,65.812,2.000,42.946,0.868,0.423
7,rf,blocked,114.130,14.554,80.589,0.453,0.563
8,rf,chrono,117.107,11.547,83.148,0.630,0.506


## Selected Hyperparameters

The hyperparameter configurations selected across folds provide clear empirical evidence of the validation schemes' behavioral differences:

- **Random Validation**: Consistently selects `min_samples_leaf=1` and `max_features=0.5` across all five folds. The absence of temporal barriers allows deep, unconstrained leaves to memorize local hour-to-hour transitions, effectively rewarding overfitting to adjacent timestamps.
- **Blocked Validation**: Shifts toward regularized structures, predominantly selecting `min_samples_leaf=5` and `max_features=1.0`. When test samples form contiguous blocks, memorizing single training points loses utility, prompting the selector to enforce smoother leaves.
- **Chronological Validation**: Consistently selects `min_samples_leaf=1` alongside `max_features=1.0`. With expanding historical horizons, the full feature set is preferred at each split, retaining fine-grained splitting capability while combating forward temporal shift.

In [8]:
rows = []
for (name, scheme), (folds, _) in results.items():
    params = pd.DataFrame([ast.literal_eval(s) for s in folds["best_params"]])
    params.columns = [c.split("__")[-1] for c in params.columns]
    rows.append(params.assign(model=name, scheme=scheme, fold=folds["fold"].to_numpy()))

pd.concat(rows).set_index(["model", "scheme", "fold"])

max_features  min_samples_leaf
model scheme  fold                                
rf    random  1              0.5                 1
              2              0.5                 1
              3              0.5                 1
              4              0.5                 1
              5              0.5                 1
      blocked 1              1.0                 5
              2              1.0                 5
              3              1.0                 1
              4              0.5                 1
              5              0.5                 5
      chrono  1              1.0                 1
              2              1.0                 1
              3              1.0                 1
              4              1.0                 1
              5              1.0                 1

## Fold-Level Performance (chronological scheme)

Examining individual fold errors under the chronological expanding-window scheme reveals pronounced temporal heterogeneity:

- **Fold 1 (Early 2012)** exhibits the lowest error ($\text{RMSE} = 105.1$, $\text{MAE} = 72.6$), benefiting from lower overall winter demand levels despite a restricted training history (2011 data only).
- **Folds 2 to 4 (Spring–Autumn 2012)** display a sharp rise in error, peaking in Fold 4 ($\text{RMSE} = 132.2$, $\text{MAE} = 94.4$). This coincides with seasonal demand surges and system growth that exceed the upper bounds recorded in the 2011 training set.
- **Fold 5 (Late 2012)** stabilizes ($\text{RMSE} = 106.0$, $\text{MAE} = 72.3$, $R^2 = 0.683$), as the expanded training window (nearly two full years) provides sufficient seasonal and weather coverage to calibrate winter commute profiles effectively.

In [9]:
for name in variants:
    print(name)
    display(results[(name, "chrono")][0].drop(columns="best_params")
            .round({"rmse": 1, "mae": 1, "r2": 3, "rmsle": 3}))

rf


,fold,n_train,n_test,test_start,test_end,rmse,mae,r2,rmsle
0,1,8657,1746,2012-01-01 12:00:00,2012-03-14 13:00:00,105.1,72.6,0.408,0.681
1,2,10403,1753,2012-03-14 14:00:00,2012-05-26 16:00:00,122.6,87.9,0.664,0.493
2,3,12156,1754,2012-05-26 17:00:00,2012-08-07 18:00:00,119.7,88.5,0.701,0.408
3,4,13910,1754,2012-08-07 19:00:00,2012-10-19 20:00:00,132.2,94.4,0.695,0.417
4,5,15664,1715,2012-10-19 21:00:00,2012-12-31 23:00:00,106.0,72.3,0.683,0.530


## Conclusions

- **Hyperparameter Selection**: The selected hyperparameters directly illustrate the mechanics of temporal leakage:
  - Under **random cross-validation**, all folds uniformly select minimal leaf sizes (`min_samples_leaf=1`) and restricted feature sampling (`max_features=0.5`), aggressively exploiting the proximity of highly correlated neighbouring hours present across splits.
  - Under **blocked validation**, the search enforces stronger smoothing (`min_samples_leaf=5` in most folds) to guard against distribution shifts across contiguous time blocks.
  - Under **chronological validation**, the selector consistently pairs `min_samples_leaf=1` with full feature availability (`max_features=1.0`), utilizing the full feature space at each split to preserve fine-grained distinctions against expanding horizons.
- **Validation Scheme Gap**: Random Forest exhibits a substantial performance drop across evaluation schemes (fold-averaged $\text{RMSE } 65.8 \text{ (random)} \rightarrow 114.1 \text{ (blocked)} \rightarrow 117.1 \text{ (chrono)}$). This proves that the optimism of random cross-validation is not specific to gradient boosting, but represents a universal challenge when evaluating flexible non-linear models on temporally autocorrelated data.
- **Extrapolation Limitations**: As with all decision tree ensembles, predictions are strictly bounded by the range of historical targets ($y \in [\min(y_{train}), \max(y_{train})]$). When ridership scaled upward in 2012, the ensemble systematically under-predicted high-volume commute peaks.
- **Scope & Constraints**: The model relies solely on static weather and calendar variables without explicit secular trend markers (`yr`) or autoregressive lag features, evaluated on a two-year observation period.